# Medical Image–Report Matching with CLIP
### Multimodal Deep Learning - Contrastive Vision-Language Learning

**Author:** Aidan Agee  
**Dataset:** [MIMIC-CXR](https://physionet.org/content/mimic-cxr/2.0.0/) (chest X-rays + radiology reports) / Synthetic demo data  
**Framework:** PyTorch + HuggingFace Transformers  
**Task:** Learn a joint embedding space where chest X-ray images and their corresponding radiology reports are closer together than non-matching pairs

---

## Overview

A radiologist writing a report about a chest X-ray is doing something remarkable - translating visual information (opacity, effusion, cardiomegaly) into natural language. This project teaches a neural network to learn that same cross-modal mapping.

We implement a **CLIP-style contrastive learning** framework that trains an image encoder and a text encoder jointly, pulling matching image-report pairs together in a shared embedding space while pushing non-matching pairs apart.

### Real-World Applications
- **Zero-shot disease classification** - classify X-rays using text prompts without labeled image data
- **Report retrieval** - given a new X-ray, find the most similar historical report
- **Image retrieval** - given a report description, find matching X-rays in a database
- **Weakly supervised detection** - localize findings using only report-level labels

### This Project Completes a Medical AI Series:
| Project | Modality | Task | Architecture |
|---|---|---|---|
| [Aneurysm Detection](https://github.com/aidanagee/aneurysm-detection-rsna) | 3D CT | Segmentation | 3D U-Net |
| [Lung Nodule Detection](https://github.com/aidanagee/lung-nodule-detection-3d-unet) | 3D CT | Segmentation | 3D U-Net |
| [ADR Detection](https://github.com/aidanagee/adr-detection-distilbert) | Text | Classification | DistilBERT |
| **Image-Report Matching (this project)** | Image + Text | Multimodal Retrieval | CLIP-style |


---
## 1. Background: Contrastive Learning and CLIP

### What is CLIP?
**CLIP** (Contrastive Language–Image Pretraining, OpenAI 2021) learns visual concepts from natural language supervision. Rather than training "image → fixed label", it trains "does this image match this text?"

This is powerful because:
- No need for expensive manual image labeling
- Learns richer representations than single-label classification
- Enables zero-shot transfer - classify images with text prompts never seen during training

### How Contrastive Learning Works

Given a batch of N image-report pairs:

```
Images:  [X-ray_1,  X-ray_2,  X-ray_3,  ... X-ray_N ]
Reports: [Report_1, Report_2, Report_3, ... Report_N ]
```

We encode both into a shared embedding space and compute an N×N similarity matrix:

```
              Report_1  Report_2  Report_3
X-ray_1   [   HIGH,     low,      low   ]   ← X-ray_1 should match Report_1
X-ray_2   [   low,      HIGH,     low   ]   ← X-ray_2 should match Report_2  
X-ray_3   [   low,      low,      HIGH  ]   ← X-ray_3 should match Report_3
```

The diagonal = correct pairs (should be HIGH similarity)
Off-diagonal = incorrect pairs (should be LOW similarity)

**InfoNCE Loss** (the contrastive loss) maximizes diagonal similarity while minimizing off-diagonal similarity - essentially an N-way classification problem in both directions.

### Why this is hard for medical images
- Medical reports use highly specialized vocabulary ("bibasilar atelectasis", "patchy consolidation")
- Visual findings can be subtle - a small pleural effusion looks similar to normal
- Same finding described differently by different radiologists
- Requires the model to learn fine-grained visual-linguistic alignment


---
## 2. Imports and Setup

In [ ]:
import os
import re
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

try:
    from transformers import (
        DistilBertTokenizer,
        DistilBertModel,
    )
    print("HuggingFace Transformers available")
except ImportError:
    print("Install: pip install transformers")

try:
    import torchvision.transforms as transforms
    import torchvision.models as models
    print("TorchVision available")
except ImportError:
    print("Install: pip install torchvision")

from PIL import Image

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

---
## 3. Dataset: MIMIC-CXR + Synthetic Demo

### About MIMIC-CXR
MIMIC-CXR contains 227,827 chest X-ray imaging studies from Beth Israel Deaconess Medical Center. Each study includes:
- One or more chest X-ray images (PA, lateral, AP views)
- A free-text radiology report written by a board-certified radiologist

Reports follow a standard structure:
- **Findings:** Detailed description of what is visible
- **Impression:** Radiologist's summary/diagnosis

### Radiology Report Vocabulary
Understanding the domain language is important for interpreting results:

| Finding | Description |
|---|---|
| Cardiomegaly | Enlarged heart shadow |
| Pleural effusion | Fluid around the lungs |
| Atelectasis | Partial lung collapse |
| Consolidation | Airspace filled with fluid (pneumonia pattern) |
| Pneumothorax | Air in pleural space |
| No acute findings | Normal or chronic changes only |

We use **synthetic data** that mirrors the structure and vocabulary of real MIMIC-CXR reports.


In [ ]:
# Synthetic radiology report templates representative of MIMIC-CXR style
REPORT_TEMPLATES = {
    'normal': [
        "The lungs are clear and well expanded. No focal consolidation, pleural effusion, or pneumothorax. The cardiomediastinal silhouette is within normal limits. No acute cardiopulmonary process.",
        "Lungs are clear bilaterally. Heart size is normal. No pleural effusion or pneumothorax identified. Bony thorax is intact. No acute findings.",
        "No acute cardiopulmonary abnormality. Clear lungs bilaterally. Normal cardiac silhouette. No effusion or pneumothorax.",
    ],
    'pneumonia': [
        "There is a focal area of consolidation in the right lower lobe, consistent with pneumonia. No pleural effusion. Heart size is normal. Recommend clinical correlation.",
        "Left lower lobe opacity consistent with pneumonia or atelectasis. No pneumothorax. Mild cardiomegaly. Clinical correlation recommended.",
        "Patchy airspace opacity in the right middle lobe suggesting pneumonia. Remainder of lungs clear. No effusion identified.",
    ],
    'effusion': [
        "Moderate left pleural effusion with associated atelectasis. No pneumothorax. Mild cardiomegaly. Right lung clear.",
        "Small bilateral pleural effusions, left greater than right. Bibasilar atelectasis. Cardiomegaly present. No pneumothorax.",
        "Large right pleural effusion causing compressive atelectasis of the right lower lobe. Left lung clear. Recommend thoracentesis consideration.",
    ],
    'cardiomegaly': [
        "Moderate cardiomegaly. No pleural effusion or pneumothorax. Lungs are clear. Vascular congestion noted. Clinical correlation with cardiac history recommended.",
        "Enlarged cardiac silhouette consistent with cardiomegaly. Mild pulmonary vascular congestion. No focal consolidation or effusion.",
        "Cardiomegaly with mild pulmonary edema pattern. Small bilateral pleural effusions. No pneumothorax.",
    ],
    'pneumothorax': [
        "Small right apical pneumothorax identified. No tension physiology. Lungs otherwise clear. Recommend follow-up imaging.",
        "Left-sided pneumothorax with approximately 20% lung collapse. No mediastinal shift. Clinical correlation and possible intervention warranted.",
        "Bilateral small pneumothoraces identified. No tension pneumothorax. Recommend urgent clinical evaluation.",
    ]
}

FINDINGS_LABELS = list(REPORT_TEMPLATES.keys())


def create_synthetic_xray(finding_type, size=(224, 224)):
    # 
    Generate a synthetic grayscale chest X-ray image.
    
    Real X-rays would be loaded from DICOM or PNG files.
    Synthetic images use structured noise patterns that vary by pathology
    type - different findings leave different visual signatures.
    # 
    h, w = size
    # Base lung field - dark (air) with some structure
    img = np.random.normal(0.2, 0.05, (h, w)).astype(np.float32)
    
    # Spine/mediastinum - brighter central stripe
    img[:, w//2-15:w//2+15] = np.random.normal(0.7, 0.05, (h, 30))
    
    # Ribs - subtle horizontal bright bands
    for i in range(0, h, 25):
        img[i:i+3, :] = np.clip(img[i:i+3, :] + 0.15, 0, 1)
    
    # Diaphragm
    img[int(0.7*h):int(0.73*h), :] = np.random.normal(0.75, 0.05, (int(0.03*h), w))
    
    # Pathology-specific modifications
    if finding_type == 'pneumonia':
        # Right lower lobe opacity
        region = img[int(0.5*h):int(0.8*h), int(0.55*w):int(0.9*w)]
        img[int(0.5*h):int(0.8*h), int(0.55*w):int(0.9*w)] =             np.clip(region + np.random.normal(0.3, 0.08, region.shape), 0, 1)
    
    elif finding_type == 'effusion':
        # Left pleural effusion - homogeneous opacity at base
        region = img[int(0.65*h):, :int(0.45*w)]
        img[int(0.65*h):, :int(0.45*w)] =             np.clip(np.random.normal(0.75, 0.03, region.shape), 0, 1)
    
    elif finding_type == 'cardiomegaly':
        # Enlarged cardiac shadow
        img[int(0.3*h):int(0.75*h), int(0.3*w):int(0.7*w)] =             np.clip(np.random.normal(0.65, 0.05, 
                (int(0.45*h), int(0.4*w))), 0, 1)
    
    elif finding_type == 'pneumothorax':
        # Right apical air - dark region
        img[:int(0.25*h), int(0.6*w):] =             np.random.normal(0.05, 0.02, (int(0.25*h), int(0.4*w)))
    
    return np.clip(img, 0, 1)


class MimicCXRDataset(Dataset):
    # 
    Dataset for chest X-ray to radiology report matching.
    
    Each sample: (image_tensor, report_text, finding_label)
    
    For real MIMIC-CXR data:
    - Images: Load from PNG files in mimic-cxr-jpg directory
    - Reports: Parse from txt files, extract Findings + Impression sections
    - Labels: From mimic-cxr-2.0.0-chexpert.csv (CheXpert automated labels)
    # 
    
    def __init__(self, n_samples=500, split='train', img_size=224):
        self.img_size = img_size
        self.split = split
        self.samples = self._generate_samples(n_samples)
        
        # Image transforms - standard for pretrained ImageNet models
        # Note: X-rays are grayscale but we convert to 3-channel for ResNet
        if split == 'train':
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(p=0.3),  # X-rays can be flipped
                transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
                transforms.ToTensor(),
                transforms.Lambda(lambda x: x.repeat(3, 1, 1)),  # grayscale → RGB
                transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                                     std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])
        
        print(f"Dataset ({split}): {len(self.samples)} samples")
        label_counts = {}
        for _, _, label in self.samples:
            label_counts[label] = label_counts.get(label, 0) + 1
        for label, count in sorted(label_counts.items()):
            print(f"  {label}: {count}")
    
    def _generate_samples(self, n):
        samples = []
        for i in range(n):
            finding = random.choice(FINDINGS_LABELS)
            report = random.choice(REPORT_TEMPLATES[finding])
            # Add some natural variation
            if random.random() > 0.7:
                report = report + " " + random.choice([
                    "No other acute findings.",
                    "Stable compared to prior.",
                    "Clinical correlation recommended.",
                ])
            samples.append((finding, report, finding))
        
        # Split
        split_idx = int(0.8 * len(samples))
        if self.split == 'train':
            return samples[:split_idx]
        return samples[split_idx:]
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        finding_type, report_text, label = self.samples[idx]
        
        # Generate synthetic X-ray image
        img_array = create_synthetic_xray(finding_type, (self.img_size, self.img_size))
        img_uint8 = (img_array * 255).astype(np.uint8)
        image = self.transform(img_uint8)
        
        return image, report_text, label


# Create datasets
train_dataset = MimicCXRDataset(n_samples=400, split='train')
val_dataset = MimicCXRDataset(n_samples=100, split='val')

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

# Visualize synthetic X-rays
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for i, finding in enumerate(FINDINGS_LABELS):
    img = create_synthetic_xray(finding)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(finding.capitalize(), fontsize=10)
    axes[i].axis('off')
plt.suptitle('Synthetic Chest X-Ray Patterns by Finding Type', fontsize=12)
plt.tight_layout()
plt.savefig('synthetic_xrays.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: synthetic_xrays.png")

---
## 4. Model Architecture: Dual Encoder with Contrastive Learning

We implement a CLIP-style dual encoder:

```
Chest X-ray ──► [Image Encoder] ──► image_embedding (256-d)
                                              │
                                    cosine similarity matrix
                                              │
Radiology Report ──► [Text Encoder] ──► text_embedding (256-d)
```

### Image Encoder
- **ResNet-50** pretrained on ImageNet (backbone)
- Final classification layer replaced with a projection head
- Outputs L2-normalized 256-dimensional embedding

### Text Encoder  
- **DistilBERT** pretrained on general text
- [CLS] token representation projected to 256-d
- Outputs L2-normalized 256-dimensional embedding

### Why the same dimensionality matters
Both encoders project to the **same 256-d space** - this is what allows cross-modal comparison. The contrastive loss trains both encoders simultaneously so that the geometry of image embeddings aligns with the geometry of text embeddings.

### Temperature parameter (τ)
A learnable temperature parameter scales the similarity logits. Lower temperature = sharper probability distribution (more confident). The model learns the optimal temperature during training.


In [ ]:
class ImageEncoder(nn.Module):
    # 
    ResNet-50 backbone with projection head for image embedding.
    
    We use a pretrained ResNet-50 and replace the final FC layer
    with a projection MLP that maps to our shared embedding space.
    
    The projection head (2-layer MLP with ReLU) is commonly used in
    contrastive learning to improve representation quality.
    # 
    
    def __init__(self, embed_dim=256, freeze_backbone=False):
        super().__init__()
        
        # Load pretrained ResNet-50
        resnet = models.resnet50(pretrained=True)
        
        # Remove final classification layer - keep feature extractor
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        backbone_dim = 2048  # ResNet-50 output dim
        
        # Optionally freeze backbone - fine-tune only projection head
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        # Projection head: backbone_dim → embed_dim
        self.projection = nn.Sequential(
            nn.Linear(backbone_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, embed_dim)
        )
    
    def forward(self, x):
        # Extract features: (B, 2048, 1, 1)
        features = self.backbone(x)
        features = features.flatten(1)  # (B, 2048)
        
        # Project to shared embedding space
        embeddings = self.projection(features)  # (B, embed_dim)
        
        # L2 normalize - cosine similarity = dot product of unit vectors
        return F.normalize(embeddings, p=2, dim=1)


class TextEncoder(nn.Module):
    # 
    DistilBERT with projection head for text embedding.
    
    Takes tokenized report text and produces a fixed-size embedding
    in the same space as the image encoder.
    
    We use the [CLS] token hidden state as the sequence representation
    - same approach as in standard BERT classification.
    # 
    
    def __init__(self, embed_dim=256, freeze_backbone=False):
        super().__init__()
        
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        bert_dim = 768  # DistilBERT hidden size
        
        if freeze_backbone:
            for param in self.bert.parameters():
                param.requires_grad = False
        
        self.projection = nn.Sequential(
            nn.Linear(bert_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, embed_dim)
        )
    
    def forward(self, input_ids, attention_mask):
        # Get BERT outputs
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # [CLS] token = outputs.last_hidden_state[:, 0, :]
        cls_output = outputs.last_hidden_state[:, 0, :]  # (B, 768)
        
        # Project to shared space
        embeddings = self.projection(cls_output)  # (B, embed_dim)
        
        return F.normalize(embeddings, p=2, dim=1)


class MedCLIP(nn.Module):
    # 
    Medical CLIP: Joint image-text contrastive learning model.
    
    Inspired by:
    - CLIP (Radford et al., OpenAI 2021)
    - CheXzero (Tiu et al., 2022) - zero-shot chest X-ray classification
    - BioViL (Bannur et al., 2023) - biomedical vision-language model
    # 
    
    def __init__(self, embed_dim=256):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim)
        self.text_encoder = TextEncoder(embed_dim)
        
        # Learnable temperature - initialized to ln(1/0.07) ≈ 2.65 (from CLIP paper)
        self.logit_scale = nn.Parameter(torch.ones([]) * math.log(1 / 0.07))
    
    def encode_image(self, images):
        return self.image_encoder(images)
    
    def encode_text(self, input_ids, attention_mask):
        return self.text_encoder(input_ids, attention_mask)
    
    def forward(self, images, input_ids, attention_mask):
        image_embeddings = self.encode_image(images)
        text_embeddings = self.encode_text(input_ids, attention_mask)
        
        # Temperature-scaled cosine similarity matrix
        # Clamp temperature to prevent training instability
        logit_scale = self.logit_scale.exp().clamp(max=100)
        
        # (B, B) similarity matrix
        logits_per_image = logit_scale * image_embeddings @ text_embeddings.T
        logits_per_text = logits_per_image.T
        
        return logits_per_image, logits_per_text, image_embeddings, text_embeddings


# Instantiate model
model = MedCLIP(embed_dim=256).to(device)

# Parameter count
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"  Image encoder: {sum(p.numel() for p in model.image_encoder.parameters()):,}")
print(f"  Text encoder:  {sum(p.numel() for p in model.text_encoder.parameters()):,}")

---
## 5. InfoNCE Contrastive Loss

**InfoNCE** (Information Noise-Contrastive Estimation) is the loss function behind CLIP. For a batch of N image-report pairs:

- Correct pairs (diagonal): image_i should match text_i
- Incorrect pairs (off-diagonal): image_i should NOT match text_j (i≠j)

This is a symmetric cross-entropy loss - we compute it in both directions (image→text and text→image) and average them.

The batch size matters a lot: larger batches = more negative examples = harder task = better representations. This is why CLIP was trained with batch sizes of 32,768.


In [ ]:
class InfoNCELoss(nn.Module):
    # 
    Symmetric InfoNCE loss for contrastive image-text learning.
    
    For a batch of N pairs, ground truth is the identity matrix -
    image_i should match text_i (diagonal = correct pairs).
    
    This is equivalent to N-way classification in both directions.
    # 
    
    def __init__(self):
        super().__init__()
    
    def forward(self, logits_per_image, logits_per_text):
        batch_size = logits_per_image.shape[0]
        
        # Ground truth: diagonal (image_i matches text_i)
        labels = torch.arange(batch_size, device=logits_per_image.device)
        
        # Symmetric loss
        loss_image = F.cross_entropy(logits_per_image, labels)
        loss_text = F.cross_entropy(logits_per_text, labels)
        
        return (loss_image + loss_text) / 2


def compute_retrieval_metrics(logits_per_image):
    # 
    Compute image-to-text retrieval accuracy.
    
    R@1: Is the correct report the top-1 result?
    R@5: Is the correct report in the top-5 results?
    
    These are standard metrics for retrieval tasks.
    # 
    batch_size = logits_per_image.shape[0]
    labels = torch.arange(batch_size, device=logits_per_image.device)
    
    # Top-1 accuracy
    top1_preds = logits_per_image.argmax(dim=1)
    r1 = (top1_preds == labels).float().mean().item()
    
    # Top-5 accuracy
    if batch_size >= 5:
        top5_preds = logits_per_image.topk(5, dim=1).indices
        r5 = sum(labels[i].item() in top5_preds[i].tolist() 
                 for i in range(batch_size)) / batch_size
    else:
        r5 = r1
    
    return {'R@1': r1, 'R@5': r5}


criterion = InfoNCELoss()

# Demonstrate loss on random embeddings
dummy_img_emb = F.normalize(torch.randn(8, 256), p=2, dim=1).to(device)
dummy_txt_emb = F.normalize(torch.randn(8, 256), p=2, dim=1).to(device)
logit_scale = math.exp(math.log(1 / 0.07))

logits_img = logit_scale * dummy_img_emb @ dummy_txt_emb.T
logits_txt = logits_img.T

loss = criterion(logits_img, logits_txt)
metrics = compute_retrieval_metrics(logits_img)

print(f"Random embeddings baseline:")
print(f"  InfoNCE Loss: {loss.item():.4f}  (expected ~ln(8) = {math.log(8):.4f} for random)")
print(f"  R@1: {metrics['R@1']:.3f}  (expected ~0.125 = 1/8 for random)")
print(f"  R@5: {metrics['R@5']:.3f}  (expected ~0.625 = 5/8 for random)")

---
## 6. Training

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def collate_fn(batch):
    # 
    Custom collate function to tokenize report text in batches.
    Tokenization is done here (not in __getitem__) for efficiency.
    # 
    images, reports, labels = zip(*batch)
    
    images = torch.stack(images)
    
    encodings = tokenizer(
        list(reports),
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    return images, encodings['input_ids'], encodings['attention_mask'], list(labels)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False,
                        collate_fn=collate_fn, num_workers=0)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_r1, total_r5 = 0.0, 0.0, 0.0
    
    for images, input_ids, attention_mask, _ in loader:
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        
        optimizer.zero_grad()
        
        logits_per_image, logits_per_text, _, _ = model(
            images, input_ids, attention_mask)
        
        loss = criterion(logits_per_image, logits_per_text)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        metrics = compute_retrieval_metrics(logits_per_image.detach())
        total_r1 += metrics['R@1']
        total_r5 += metrics['R@5']
    
    n = len(loader)
    return {'loss': total_loss/n, 'R@1': total_r1/n, 'R@5': total_r5/n}


def validate(model, loader, criterion, device):
    model.eval()
    total_loss, total_r1, total_r5 = 0.0, 0.0, 0.0
    
    with torch.no_grad():
        for images, input_ids, attention_mask, _ in loader:
            images = images.to(device)
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            
            logits_per_image, logits_per_text, _, _ = model(
                images, input_ids, attention_mask)
            
            loss = criterion(logits_per_image, logits_per_text)
            total_loss += loss.item()
            
            metrics = compute_retrieval_metrics(logits_per_image)
            total_r1 += metrics['R@1']
            total_r5 += metrics['R@5']
    
    n = len(loader)
    return {'loss': total_loss/n, 'R@1': total_r1/n, 'R@5': total_r5/n}


# Training
N_EPOCHS = 8
optimizer = torch.optim.AdamW([
    {'params': model.image_encoder.backbone.parameters(), 'lr': 1e-5},
    {'params': model.image_encoder.projection.parameters(), 'lr': 1e-4},
    {'params': model.text_encoder.bert.parameters(), 'lr': 1e-5},
    {'params': model.text_encoder.projection.parameters(), 'lr': 1e-4},
    {'params': [model.logit_scale], 'lr': 1e-3},
], weight_decay=0.01)

# Different LRs for backbone vs projection head:
# Pretrained weights need gentle fine-tuning (1e-5)
# Randomly initialized projection heads can use larger LR (1e-4)
print("Using differential learning rates:")
print("  Backbone layers (pretrained): lr=1e-5")
print("  Projection heads (random init): lr=1e-4")
print()

history = {k: [] for k in ['train_loss','val_loss','train_r1','val_r1','train_r5','val_r5']}
best_val_r1 = 0.0
best_state = None

print(f"{'Epoch':>6} {'Tr Loss':>9} {'Val Loss':>9} {'Tr R@1':>8} {'Val R@1':>8} {'Val R@5':>8} {'Temp':>7}")
print("-" * 65)

for epoch in range(1, N_EPOCHS + 1):
    train_m = train_epoch(model, train_loader, optimizer, criterion, device)
    val_m = validate(model, val_loader, criterion, device)
    
    for split, m in [('train', train_m), ('val', val_m)]:
        history[f'{split}_loss'].append(m['loss'])
        history[f'{split}_r1'].append(m['R@1'])
        history[f'{split}_r5'].append(m['R@5'])
    
    temp = model.logit_scale.exp().item()
    print(f"{epoch:>6} {train_m['loss']:>9.4f} {val_m['loss']:>9.4f} "
          f"{train_m['R@1']:>8.3f} {val_m['R@1']:>8.3f} {val_m['R@5']:>8.3f} {temp:>7.2f}")
    
    if val_m['R@1'] > best_val_r1:
        best_val_r1 = val_m['R@1']
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

if best_state:
    model.load_state_dict(best_state)
    print(f"\nRestored best model (val R@1: {best_val_r1:.3f})")

---
## 7. Embedding Space Visualization & Retrieval Demo

In [ ]:
def plot_training_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train', markersize=4)
    axes[0].plot(epochs, history['val_loss'], 'r-o', label='Val', markersize=4)
    axes[0].set_title('InfoNCE Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs, history['train_r1'], 'b-o', label='Train R@1', markersize=4)
    axes[1].plot(epochs, history['val_r1'], 'r-o', label='Val R@1', markersize=4)
    axes[1].set_title('Recall@1 (Top-1 Retrieval Accuracy)')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(epochs, history['val_r1'], 'r-o', label='R@1', markersize=4)
    axes[2].plot(epochs, history['val_r5'], 'g-s', label='R@5', markersize=4)
    axes[2].set_title('Val Retrieval Metrics')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle('MedCLIP Training - Chest X-Ray to Report Matching', fontsize=12)
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()


def visualize_similarity_matrix(model, dataset, tokenizer, device, n=6):
    # 
    Visualize the N x N similarity matrix for a sample batch.
    Diagonal should be bright (correct pairs), off-diagonal dark.
    # 
    model.eval()
    
    indices = random.sample(range(len(dataset)), n)
    images, reports, labels = [], [], []
    
    for i in indices:
        img, report, label = dataset[i]
        images.append(img)
        reports.append(report)
        labels.append(label)
    
    images = torch.stack(images).to(device)
    encodings = tokenizer(reports, max_length=128, padding='max_length',
                          truncation=True, return_tensors='pt')
    
    with torch.no_grad():
        logits, _, img_emb, txt_emb = model(
            images,
            encodings['input_ids'].to(device),
            encodings['attention_mask'].to(device)
        )
        sim_matrix = (img_emb @ txt_emb.T).cpu().numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Similarity matrix
    im = axes[0].imshow(sim_matrix, cmap='Blues', vmin=-1, vmax=1)
    axes[0].set_title('Image-Text Cosine Similarity Matrix\n(diagonal = correct pairs)')
    axes[0].set_xlabel('Report index')
    axes[0].set_ylabel('X-ray index')
    axes[0].set_xticks(range(n))
    axes[0].set_yticks(range(n))
    axes[0].set_xticklabels([l[:8] for l in labels], rotation=45, ha='right')
    axes[0].set_yticklabels(labels)
    
    for i in range(n):
        for j in range(n):
            color = 'white' if sim_matrix[i,j] > 0.3 else 'black'
            axes[0].text(j, i, f'{sim_matrix[i,j]:.2f}', 
                        ha='center', va='center', fontsize=8, color=color)
    plt.colorbar(im, ax=axes[0])
    
    # Sample X-ray images
    axes[1].axis('off')
    sample_img = images[0].cpu()
    # Denormalize
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    sample_img = (sample_img * std + mean).clamp(0, 1)
    axes[1].imshow(sample_img.permute(1,2,0).numpy(), cmap='gray')
    axes[1].set_title(f'Sample X-ray: {labels[0]}\n"{reports[0][:80]}..."')
    
    plt.suptitle('MedCLIP Embedding Space Analysis', fontsize=12)
    plt.tight_layout()
    plt.savefig('similarity_matrix.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved: similarity_matrix.png")


plot_training_history(history)
visualize_similarity_matrix(model, val_dataset, tokenizer, device)

---
## 8. Zero-Shot Classification

One of CLIP's most powerful capabilities - classify images using text prompts **without any labeled image training data**. We define class descriptions as text and find which text embedding is most similar to a given image embedding.

This is how CheXzero (Tiu et al., 2022) achieved radiologist-level chest X-ray diagnosis using only report supervision.


In [ ]:
def zero_shot_classify(model, tokenizer, image, class_prompts, device):
    # 
    Classify an image using text prompts - no labeled image data needed.
    
    Args:
        class_prompts: dict of {class_name: text_description}
    
    Returns:
        Predicted class and probability distribution
    # 
    model.eval()
    
    # Encode image
    img_tensor = image.unsqueeze(0).to(device)
    with torch.no_grad():
        img_emb = model.encode_image(img_tensor)
    
    # Encode all class text prompts
    class_names = list(class_prompts.keys())
    texts = list(class_prompts.values())
    
    encodings = tokenizer(texts, max_length=128, padding='max_length',
                          truncation=True, return_tensors='pt')
    
    with torch.no_grad():
        txt_emb = model.encode_text(
            encodings['input_ids'].to(device),
            encodings['attention_mask'].to(device)
        )
    
    # Similarity scores
    logit_scale = model.logit_scale.exp().clamp(max=100)
    similarities = (logit_scale * img_emb @ txt_emb.T).squeeze(0)
    probs = F.softmax(similarities, dim=0).cpu().numpy()
    
    predicted_class = class_names[probs.argmax()]
    return predicted_class, dict(zip(class_names, probs))


# Define zero-shot prompts - clinical descriptions of each finding
ZERO_SHOT_PROMPTS = {
    'Normal': "No acute cardiopulmonary process. Clear lungs bilaterally. Normal cardiac silhouette.",
    'Pneumonia': "Focal consolidation or airspace opacity consistent with pneumonia. Lobar or segmental distribution.",
    'Pleural Effusion': "Pleural effusion with blunting of costophrenic angle. Fluid in pleural space.",
    'Cardiomegaly': "Enlarged cardiac silhouette. Cardiomegaly present. Increased cardiothoracic ratio.",
    'Pneumothorax': "Pneumothorax with visible lung edge. Air in pleural space. Lung collapse."
}

# Test on sample images
print("ZERO-SHOT CLASSIFICATION DEMO")
print("=" * 55)

for finding in FINDINGS_LABELS:
    img_array = create_synthetic_xray(finding)
    img_uint8 = (img_array * 255).astype(np.uint8)
    
    # Apply val transform
    transform = val_dataset.transform
    image = transform(img_uint8)
    
    pred, probs = zero_shot_classify(model, tokenizer, image, ZERO_SHOT_PROMPTS, device)
    
    print(f"\nTrue: {finding.upper()}")
    print(f"Pred: {pred}")
    print("Probabilities:")
    for cls, p in sorted(probs.items(), key=lambda x: -x[1]):
        bar = "█" * int(p * 30)
        marker = " ←" if cls.lower() == finding.lower() or                   (finding == 'effusion' and 'Effusion' in cls) else ""
        print(f"  {cls:20s} {p:.3f} {bar}{marker}")

---
## 9. Discussion & Clinical Relevance

### What this model learns
Through contrastive training, the model develops a shared embedding space where:
- X-rays showing pneumonia cluster near reports describing "consolidation" and "opacity"
- Normal X-rays cluster near reports describing "clear lungs" and "no acute findings"
- Similar pathologies are nearby even across different patients and scanners

### Limitations
1. **Synthetic data** - real clinical deployment requires MIMIC-CXR (requires PhysioNet credentialing)
2. **Batch size constraint** - CLIP-style training benefits enormously from large batches; we use 16 due to memory limits vs CLIP's 32,768
3. **Report quality** - real reports have significant radiologist-to-radiologist variability
4. **Multi-label complexity** - patients often have multiple findings simultaneously; binary matching oversimplifies this

### Real-World Extensions
- **BioViL** (Microsoft, 2023) - adds phrase grounding to localize findings
- **CheXzero** - zero-shot diagnosis at radiologist level using only report supervision
- **MedCLIP** (Wang et al., 2022) - decoupled contrastive learning for medical vision-language

### Clinical Impact
Systems like this could:
- Assist radiologists by retrieving similar historical cases with known outcomes
- Enable rapid screening in resource-limited settings
- Power automated quality checks on report completeness

---
## References

1. Radford, A., et al. (2021). Learning Transferable Visual Models from Natural Language Supervision (CLIP). *ICML 2021.*
2. Tiu, E., et al. (2022). Expert-level detection of pathologies from unannotated chest X-ray images via self-supervised learning (CheXzero). *Nature Biomedical Engineering.*
3. Wang, Z., et al. (2022). MedCLIP: Contrastive Learning from Unpaired Medical Images and Text. *EMNLP 2022.*
4. Bannur, S., et al. (2023). Learning to Exploit Temporal Structure for Biomedical Vision-Language Processing (BioViL). *CVPR 2023.*
5. Johnson, A., et al. (2019). MIMIC-CXR: A large publicly available database of labeled chest radiographs. *PhysioNet.*
